In [32]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

class UPSCState(TypedDict):
    essay : str
    language_feedback : str
    clarity_feedback : str
    analysis_feedback : str
    overall_feeback : str
    individual_scores : Annotated[list[int], operator.add]
    avg_score : float

class EvaluationSchema(BaseModel):
    feedback : str = Field(description = "Detailed Feedback for the essay")
    score : float = Field(description = "Score out of 10 ", ge = 0, le = 10)

In [8]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model = 'gpt-5', temperature =0)
structure_model = llm.with_structured_output(EvaluationSchema)

text = ''

llm_response = llm.invoke(f"Evaluate the language quality from the essay and assign the score out of 10 to it{'''Demonetization is the act of stripping a currency unit of its status as legal tender by the government or central bank. In India, this historic move took place on November 8, 2016, when Prime Minister Narendra Modi announced that ₹500 and ₹1,000 notes—which made up about 86% of the country’s cash supply—would no longer be valid.
The primary goals were to eliminate black money, curb corruption, stop counterfeit currency, and crack down on terror funding while pushing the nation toward a digital, cashless economy.
The decision had mixed results. On the positive side, it boosted digital payments, increased formal banking deposits, and improved tax compliance. However, it caused severe short-term hardships, including long bank queues, cash shortages, and economic slowdowns for small businesses and the rural agricultural sector. While supporters viewed it as a bold strike against financial crime, critics argued that much of the unaccounted money ultimately returned to the system, making its long-term impact a subject of ongoing debate'''}").content

structure_response = structure_model.invoke('''Demonetization is the act of stripping a currency unit of its status as legal tender by the government or central bank. In India, this historic move took place on November 8, 2016, when Prime Minister Narendra Modi announced that ₹500 and ₹1,000 notes—which made up about 86% of the country’s cash supply—would no longer be valid.
The primary goals were to eliminate black money, curb corruption, stop counterfeit currency, and crack down on terror funding while pushing the nation toward a digital, cashless economy.
The decision had mixed results. On the positive side, it boosted digital payments, increased formal banking deposits, and improved tax compliance. However, it caused severe short-term hardships, including long bank queues, cash shortages, and economic slowdowns for small businesses and the rural agricultural sector. While supporters viewed it as a bold strike against financial crime, critics argued that much of the unaccounted money ultimately returned to the system, making its long-term impact a subject of ongoing debate
evaluate the language quality from the above essay and assign the score out of 10''')

print(llm_response)
print()
print(structure_response)

Language quality evaluation:

Strengths:
- Clear, concise, and coherent; maintains a professional, neutral tone.
- Strong structure: definition, context, goals, outcomes, and perspectives.
- Grammatically sound with precise vocabulary and varied sentence structure.
- Effective transitions and balanced phrasing (“supporters viewed… critics argued…”).

Areas to improve:
- Add a brief concluding sentence to neatly close the discussion.
- Slightly vary phrasing to avoid repetition (e.g., “The outcomes were mixed” instead of “The decision had mixed results”).
- Consider one or two more connective phrases for flow between the goals and outcomes.

Score: 9/10

feedback='Overall, the language is clear, concise, and well-organized. The essay defines the concept, states objectives, and evaluates outcomes in a logical sequence. Vocabulary is appropriate and precise, tone is balanced and neutral, and grammar and punctuation are correct (effective use of em dashes and parallel structure). Transitio

In [33]:
structure_response.score

9.0

In [34]:
# define the nodes
def evaluate_language(state : UPSCState):
    prompt = f"Evaluate the language quality in the given essay {state['essay']} and assign the score out of 10"
    output = structure_model.invoke(prompt)
    return {"language_feedback" : output.feedback, "individual_score" : [output.score]}

In [35]:
def evaluate_depth(state : UPSCState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign the score out of 10 \n {state['essay']}'
    output = structure_model.invoke(prompt)

    return {"analysis_feedback" : output.feedback, "individual_score" : [output.score]}

In [36]:
def evaluate_clarity(state : UPSCState):
    prompt = f'Evaluate the clarity of thoughts of the following essay and provide a feedback and assign the score out of 10 \n {state['essay']}'
    output = structure_model.invoke(prompt)

    return {"clarity_feedback" : output.feedback, "individual_score" : [output.score]}

In [37]:
def final_evaluation(state : UPSCState):
    prompt = f'''Based on the following feedback create a summarized feedback \n language Feedback {state['language_feedback']}\n
    analysis feedback {state['analysis_feedback']} \n
    clarity_feedback {state['clarity_feedback']}'''

    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {"overall_feedback" : output.feedback, "avg_score" : avg_score}

In [38]:
from langgraph.graph import StateGraph, START, END

In [42]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language" , evaluate_language)
graph.add_node("evaluate_depth", evaluate_depth)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("final_evaluation", final_evaluation)


graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_depth")
graph.add_edge(START, "evaluate_clarity")


graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_depth", "final_evaluation")
graph.add_edge("evaluate_clarity", "final_evaluation")

graph.add_edge("final_evaluation", END)
workflow = graph.compile()

In [43]:
text = '''Demonetization is the act of stripping a currency unit of its status as legal tender by the government or central bank. In India, this historic move took place on November 8, 2016, when Prime Minister Narendra Modi announced that ₹500 and ₹1,000 notes—which made up about 86% of the country’s cash supply—would no longer be valid.
The primary goals were to eliminate black money, curb corruption, stop counterfeit currency, and crack down on terror funding while pushing the nation toward a digital, cashless economy.
The decision had mixed results. On the positive side, it boosted digital payments, increased formal banking deposits, and improved tax compliance. However, it caused severe short-term hardships, including long bank queues, cash shortages, and economic slowdowns for small businesses and the rural agricultural sector. While supporters viewed it as a bold strike against financial crime, critics argued that much of the unaccounted money ultimately returned to the system, making its long-term impact a subject of ongoing debate'''

In [44]:
initial_state = {"essay" : text}
final_state = workflow.invoke(initial_state)
print(final_state)

ZeroDivisionError: division by zero